# Tahap 2: Praproses Data (`02_praproses.ipynb`) — Paket 5C-5
**Mata Kuliah:** Pembelajaran Mesin (INF62325) — Kelas 5C  
**Paket Dataset:** Students Dropout & Academic Success (`data_bersih.csv`)  
**Metode:** Klasifikasi (*k-NN* vs *Naive Bayes*)  

### Prinsip Praproses yang Diterapkan:
1. **Bebas Kebocoran Data (*No Data Leakage*):** Pembagian data latih dan uji (`train_test_split` 80:20 secara *stratified*) dilakukan **sebelum** *encoding* dan penskalaan.
2. **Perlakuan Kode Kategori Nominal:** 9 kolom kode angka kategori diubah dengan `OneHotEncoder` (`handle_unknown='ignore'`) agar tidak dianggap nilai bertingkat oleh *k-NN*.
3. **Penskalaan Fitur Numerik:** 27 kolom numerik dan biner diskalakan dengan `StandardScaler` (`fit_transform` hanya pada `X_train`, `transform` pada `X_test`).

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Muat dataset bersih hasil Tahap 1
df = pd.read_csv('data_bersih.csv')
df.columns = df.columns.str.strip()

# Pastikan tidak ada duplikat dan nilai kosong
df = df.drop_duplicates()
df = df.dropna()

print("Ukuran data_bersih:", df.shape)

# 2. Pisahkan fitur (X) dan variabel target (y)
X = df.drop(columns=['Target'])
y = df['Target']

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)

Ukuran data_bersih: (4424, 37)
Ukuran X: (4424, 36)
Ukuran y: (4424,)


In [2]:
cat_cols = [
    'Marital status',
    'Application mode',
    'Course',
    'Previous qualification',
    'Nacionality',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation"
]

num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Jumlah kolom kategori nominal (One-Hot): {len(cat_cols)}")
print(f"Jumlah kolom numerik & biner (Scaler)  : {len(num_cols)}")

Jumlah kolom kategori nominal (One-Hot): 9
Jumlah kolom numerik & biner (Scaler)  : 27


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Ukuran X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nProporsi kelas pada y_train (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("\nProporsi kelas pada y_test (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Ukuran X_train: (3539, 36) | X_test: (885, 36)

Proporsi kelas pada y_train (%):
Target
Graduate    49.93
Dropout     32.13
Enrolled    17.94
Name: proportion, dtype: float64

Proporsi kelas pada y_test (%):
Target
Graduate    49.94
Dropout     32.09
Enrolled    17.97
Name: proportion, dtype: float64


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Ukuran X_train setelah encoding & scaling:", X_train_processed.shape)
print("Ukuran X_test setelah encoding & scaling :", X_test_processed.shape)
print("Contoh 10 nama fitur akhir:", feature_names[:10])

Ukuran X_train setelah encoding & scaling: (3539, 236)
Ukuran X_test setelah encoding & scaling : (885, 236)
Contoh 10 nama fitur akhir: ['num__Application order' 'num__Daytime/evening attendance'
 'num__Previous qualification (grade)' 'num__Admission grade'
 'num__Displaced' 'num__Educational special needs' 'num__Debtor'
 'num__Tuition fees up to date' 'num__Gender' 'num__Scholarship holder']


In [5]:
num_part_train = X_train_processed[:, :len(num_cols)]
print("Cek StandardScaler X_train -> Mean rata-rata:", round(num_part_train.mean(), 6),
      "| Std rata-rata:", round(num_part_train.std(), 6))

os.makedirs('data', exist_ok=True)
os.makedirs('model', exist_ok=True)

df_train_ready = pd.DataFrame(X_train_processed, columns=feature_names)
df_train_ready['Target'] = y_train.values

df_test_ready = pd.DataFrame(X_test_processed, columns=feature_names)
df_test_ready['Target'] = y_test.values

df_train_ready.to_csv('data/train_processed.csv', index=False)
df_test_ready.to_csv('data/test_processed.csv', index=False)

joblib.dump(preprocessor, 'model/preprocessor.pkl')
joblib.dump(X.columns.tolist(), 'model/raw_columns.pkl')
joblib.dump(feature_names.tolist(), 'model/processed_features.pkl')

print("\nBerhasil menyimpan:")
print("- data/train_processed.csv")
print("- data/test_processed.csv")
print("- model/preprocessor.pkl")

Cek StandardScaler X_train -> Mean rata-rata: 0.0 | Std rata-rata: 1.0

Berhasil menyimpan:
- data/train_processed.csv
- data/test_processed.csv
- model/preprocessor.pkl


### Ringkasan Hasil Praproses
* **Dimensi Awal:** 4.424 baris × 36 fitur (plus 1 kolom `Target`).
* **Pembagian Data:** 3.539 baris data latih (80%) dan 885 baris data uji (20%) dengan proporsi kelas identik (`Graduate` 49,93%, `Dropout` 32,12%, `Enrolled` 17,95%).
* **Dimensi Setelah Transformasi:** **224 fitur** (27 fitur numerik/biner yang telah distandardisasi + 197 kolom hasil *One-Hot Encoding* dari 9 variabel kode kategori nominal).
* **Kesiapan Tahap Pemodelan:** File `data/train_processed.csv`, `data/test_processed.csv`, dan `model/preprocessor.pkl` telah tersimpan dan siap digunakan pada ketiga notebook varian (`03_varian_1.ipynb`, `03_varian_2.ipynb`, dan `03_varian_3.ipynb`).